In [ ]:
# Header for the notebook
from datetime import datetime
from IPython.display import display, Markdown

# Get the current date
title = "Template for a Jupyter Notebook"
current_date = datetime.now().strftime("%d %B %Y, %H:%M:%S")
authors = "Estelle Thomas (with Copilot & Claude)"

# Insert the date into the notebook
display(Markdown(f"# {title}"))
display(Markdown(f"{current_date}"))
display(Markdown(f"by {authors}"))

# Template for a Jupyter Notebook

09 October 2025, 10:03:37

by Denis Mottet (and Copilot)


This Jupyter notebook is a template with a header and footer.

In the header, it is included:
- Title
- Author
- Date

In the footer, it is included:
- How to produce an HTML version of the notebook and open it in a web browser.
  - The HTML version does not show cells tagged with "hide" (e.g., the cell above, and the last cell).



# Do something here 
We will use numpy and matplotlib for plotting a simple graph.


In [ ]:
# Installer openpyxl pour pouvoir exporter le dataframe dans un fichier excel
#!pip install openpyxl

# Essai de créer un dataframe avec les indices de variabilité cardiaque pour tous les sujets et les différentes conditions
# Import des packages nécessaires
import pandas as pd
import os
import neurokit2
import numpy as np

# Exclusion participants : P15
EXCLUS_HRV = {"15"}

# Dossiers avec les intervalles RR dans data/T0 et data/T1
data_dirs = ["data/T0", "data/T1"]

results = []

for data_dir in data_dirs:
    for filename in os.listdir(data_dir):
        if filename.endswith(".txt"):
        # Charger les intervalles RR
            rr = np.loadtxt(os.path.join(data_dir, filename))

        # Clean the RR intervals (optional, but recommended)
        # Remove artifacts beats (physiological implausible intervals) IBI
            rr_clean = rr[(rr > 300) & (rr < 1500)]

        # Convertir les intervalles RR en peaks
            peaks = neurokit2.intervals_to_peaks(rr_clean, sampling_rate=1000)
        
        # Calculer HRV (indices de variabilité cardiaque)
            hrv = neurokit2.hrv(peaks, sampling_rate=1000)
        
        # Extraire ID et Temps depuis le nom du fichier
            id = filename.split('_')[0]  # ID du sujet
            time = filename.split('_')[1].split('.')[0]  # Temps (T0, T1, etc.)

        results.append({
            'ID': id,
            'Temps': time,
            'HRV_RMSSD': hrv['HRV_RMSSD'].values[0],
            'HRV_HF': hrv['HRV_HF'].values[0],
            'HRV_LF': hrv['HRV_LF'].values[0],
            'HRV_LFHF': hrv['HRV_LFHF'].values[0]
        })
       
# Créer un DataFrame à partir des résultats
hrv_df = pd.DataFrame(results)
hrv_analyses = hrv_df[~hrv_df['ID'].isin(EXCLUS_HRV)]  
print(hrv_analyses)

# Export du DataFrame dans un fichier excel : HRV_results.xlsx
hrv_analyses.to_excel("HRV_results.xlsx", index=False)



In [ ]:
# https://nbconvert.readthedocs.io/en/latest/removing_cells.html
# https://github.com/msm1089/ipynbname/issues/17#issuecomment-1293269863


from traitlets.config import Config
from nbconvert.exporters import HTMLExporter
from nbconvert.preprocessors import TagRemovePreprocessor
from IPython.core.getipython import get_ipython
import os


def get_notebook_name():
    """
    Get the current notebook name (without extension).
    """
    ip = get_ipython()
    path = None
    if "__vsc_ipynb_file__" in ip.user_ns:  # type: ignore
        path = ip.user_ns["__vsc_ipynb_file__"] # type: ignore
        # split the path to get only the file name without extension

    return os.path.splitext(path)[0]  # type: ignore


# Get the notebook name
notebook_file_name = get_notebook_name()

# Switch to inline backend for HTML export (interactive backends don't work in static HTML)
get_ipython().run_line_magic("matplotlib", "inline")  # type: ignore
print("Switched to 'inline' backend for HTML export")


# Setup config
c = Config()

# Configure tag removal - be sure to tag your cells to remove  using the
# words remove_cell to remove cells. You can also modify the code to use
# a different tag word
c.TagRemovePreprocessor.remove_cell_tags = ("remove",)
c.TagRemovePreprocessor.remove_all_outputs_tags = ("remove_output",)
c.TagRemovePreprocessor.remove_input_tags = ("hide",)
c.TagRemovePreprocessor.enabled = True
c.HTMLExporter.preprocessors = ["nbconvert.preprocessors.TagRemovePreprocessor"]

# ensure the graphics are included in the html
c.HTMLExporter.embed_images = True
# do not show the input code cells (distracts from the output)
c.HTMLExporter.exclude_output_prompt = True
c.HTMLExporter.exclude_input_prompt = True

# Configure the exporter
exporter = HTMLExporter(config=c)
exporter.register_preprocessor(TagRemovePreprocessor(config=c), True)


# run our exporter - returns a tuple - first element with html,
# second with notebook metadata
output = HTMLExporter(config=c).from_filename(notebook_file_name + ".ipynb")

# Write to output html file
with open(notebook_file_name + ".html", "w") as f:
    f.write(output[0])

# open the file with the operating system's default program
# handle spaces in filename with quotes
try:
    if os.name == "posix":
        if os.uname().sysname == "Darwin":
            # macOS - use single quotes for spaces
            os.system(f"open '{notebook_file_name}.html'")
        else:
            # Linux - use single quotes for spaces
            os.system(f"xdg-open '{notebook_file_name}.html'")
    elif os.name == "nt":
        # Windows - use double quotes for empty title + for file name with spaces
        os.system(f'start "" "{notebook_file_name}.html"')
    else:
        print("Unsupported OS")
except Exception as e:
    print("Error opening file: ", e)
